# Demo LiteLLM — Clase 01

Este notebook permite verificar, de forma sencilla y paso a paso, que una instancia local de **LiteLLM** está configurada y funcionando correctamente.

## Objetivos de aprendizaje

Al completar este notebook, podrás:

1. **Cargar credenciales** de forma segura desde un archivo `.env`.
2. **Verificar la conectividad** al proxy LiteLLM usando el endpoint `/health`.
3. **Listar los modelos disponibles** en el proxy mediante `/v1/models`.
4. **Realizar una llamada de inferencia** con `gpt-4.1-mini` usando el SDK `openai`.
5. **Interpretar un error real** cuando las credenciales son incorrectas.

## ¿Qué es LiteLLM?

**LiteLLM** es un proxy de código abierto que actúa como capa de abstracción entre tu código y múltiples proveedores de LLMs (OpenAI, Anthropic, Google, etc.). En lugar de conectarte directamente a cada proveedor con sus propias APIs y formatos, tu código siempre habla con LiteLLM usando el formato estándar de OpenAI.

Esto significa que:

- **Un solo SDK** (`openai`) funciona con todos los modelos configurados en el proxy.
- Las **credenciales reales** de los proveedores quedan en el servidor del proxy, nunca en tu código.
- Puedes **cambiar de modelo** en una sola línea sin reescribir la lógica de tu aplicación.

Durante este curso, todas las llamadas a modelos pasarán por el proxy LiteLLM que ya está configurado para ti.

## Configuración

La celda siguiente carga las credenciales necesarias para conectarse al proxy LiteLLM desde el archivo `.env` ubicado en la raíz del repositorio (`.env`, junto a este notebook).

Este archivo contiene dos variables:

- **`LLM_SERVICE_URL`**: la URL base del proxy (por ejemplo, `http://localhost:4000`).
- **`LLM_API_KEY`**: la clave de acceso al proxy (`LITELLM_MASTER_KEY`).

Si alguna variable falta o está vacía, la celda lanza un `RuntimeError` con instrucciones claras para resolverlo. Esto evita ejecutar el resto del notebook con credenciales incompletas.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

ENV_PATH = Path(".env")

load_dotenv(dotenv_path=ENV_PATH)

LLM_SERVICE_URL = os.environ.get("LLM_SERVICE_URL", "")
LLM_API_KEY = os.environ.get("LLM_API_KEY", "")

for var_name, var_value in (("LLM_SERVICE_URL", LLM_SERVICE_URL), ("LLM_API_KEY", LLM_API_KEY)):
    if not var_value:
        raise RuntimeError(
            f"Variable de entorno requerida ausente o vacía: {var_name}. "
            f"Agrégala al archivo: {ENV_PATH.resolve()}"
        )

print(f"LLM_SERVICE_URL presente: {bool(LLM_SERVICE_URL)}")
print(f"LLM_API_KEY presente: {bool(LLM_API_KEY)}")

## Verificación de conexión

Antes de hacer cualquier llamada al modelo, conviene confirmar que el proxy LiteLLM está activo y accesible. La celda siguiente realiza un `GET` al endpoint `/health` del proxy, que responde con un estado simple sin necesidad de autenticación.

Verificar la conexión primero ahorra tiempo: si el proxy no está corriendo, sabrás exactamente dónde está el problema antes de intentar una inferencia completa.

In [ ]:
import time

import requests

try:
    start = time.monotonic()
    response = requests.get(f"{LLM_SERVICE_URL}/health", timeout=5)
    elapsed_ms = int((time.monotonic() - start) * 1000)
    if response.status_code == 200:
        print(f"[OK] {elapsed_ms}ms")
    else:
        print(f"[ERROR] HTTP {response.status_code}: {response.text[:200]}")
except Exception as exc:
    print(f"[ERROR] {type(exc).__name__}: {exc}")

## Modelos disponibles

El endpoint `/v1/models` lista todos los modelos que el proxy tiene configurados. Este listado es útil para confirmar qué modelos puedes usar antes de escribir código que los llame.

A diferencia del health check, este endpoint requiere autenticación con tu `LLM_API_KEY` en el encabezado `Authorization`.

In [ ]:
try:
    response = requests.get(
        f"{LLM_SERVICE_URL}/v1/models",
        headers={"Authorization": f"Bearer {LLM_API_KEY}"},
        timeout=5,
    )
    if response.status_code == 200:
        models = response.json().get("data", [])
        print("[OK] Modelos disponibles:")
        for model in models:
            print(f"  - {model['id']}")
    else:
        print(f"[ERROR] HTTP {response.status_code}: {response.text[:200]}")
except Exception as exc:
    print(f"[ERROR] {type(exc).__name__}: {exc}")

## Primera llamada al modelo

Ahora haremos la primera llamada real de inferencia. El SDK `openai` se configura con la URL del proxy y la API key, y luego usamos `chat.completions.create()` exactamente como si hablaras directamente con OpenAI, pero a través de LiteLLM.

El patrón es siempre el mismo:
1. Crear un cliente `openai.OpenAI` apuntando al proxy.
2. Llamar a `chat.completions.create()` con el modelo, los mensajes y los parámetros.
3. Leer el texto de respuesta en `response.choices[0].message.content`.

Este patrón se repetirá en todos los notebooks del curso.

In [ ]:
import openai

client = openai.OpenAI(base_url=LLM_SERVICE_URL, api_key=LLM_API_KEY)

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {
            "role": "user",
            "content": "Hola, ¿puedes confirmar que estás funcionando correctamente? Responde en una oración.",
        }
    ],
    temperature=0.7,
    max_tokens=100,
)
print(f"[OK] {response.choices[0].message.content}")

## ¿Qué pasa cuando algo falla?

Es importante saber cómo se ve un error real. La celda siguiente hace exactamente la misma llamada que la anterior, pero usa una API key intencionalmente inválida. El modelo rechazará la solicitud con un error de autenticación.

Capturamos la excepción y la mostramos con el prefijo `[ERROR]` para que puedas reconocer este tipo de fallo cuando ocurra en otros contextos. La celda termina normalmente sin interrumpir el notebook.

In [ ]:
invalid_client = openai.OpenAI(base_url=LLM_SERVICE_URL, api_key="sk-invalid-key-demo")

try:
    invalid_client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {
                "role": "user",
                "content": "Hola, ¿puedes confirmar que estás funcionando correctamente? Responde en una oración.",
            }
        ],
        temperature=0.7,
        max_tokens=100,
    )
except Exception as exc:
    print(f"[ERROR] {type(exc).__name__}: {exc}")

## Conclusión y próximos pasos

¡Completaste la verificación de tu entorno LiteLLM! En este notebook confirmaste que:

1. **Las credenciales** se cargan correctamente desde el archivo `.env`.
2. **El proxy LiteLLM** está activo y responde al health check.
3. **Los modelos disponibles** son accesibles a través de `/v1/models`.
4. **La inferencia con `gpt-4.1-mini`** funciona de extremo a extremo usando el SDK `openai`.
5. **Los errores de autenticación** se ven como excepciones del tipo `AuthenticationError` u similar.

### ¿Qué sigue?

En los próximos notebooks explorarás:

- **Clase 02**: Modos de fallo comunes de los LLMs — alucinaciones, límites de contexto, inyección de prompts, y más.
- **Clase 03**: Parámetros del modelo — cómo `temperature`, `max_tokens` y otros parámetros afectan la calidad y el comportamiento de las respuestas.